
# Chapter 3 calibration results notebook

This notebook rebuilds the calibration figures for the thesis chapter using the `calibration_results_reg_*.xlsx` workbooks.

The notebook is organized to match the final section structure:

1. justify the regularisation choice for `lambda_TS`,
2. produce time-series diagnostics for RMSE, MAE, and within-spread proportion,
3. produce cross-sectional diagnostics by equal-count maturity and moneyness buckets,
4. inspect identification issues and parameter stability for the SVCJ model.


In [1]:

from __future__ import annotations

from pathlib import Path
import math
import re
import warnings

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

# ---- configuration ----
FINAL_REG = 100
FILE_PATTERN = "calibration_results_reg_*.xlsx"

SEARCH_ROOTS = [
    Path.cwd(),
    Path.cwd().parent / "excel files",
]

OUTPUT_DIR = Path.cwd() / "chapter3_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_LABELS = {"black": "Black", "heston": "Heston", "svcj": "SVCJ"}
MODEL_COLORS = {"black": "#6b7280", "heston": "#2563eb", "svcj": "#dc2626"}

pio.templates.default = "plotly_white"


In [2]:

def find_calibration_files(pattern: str = FILE_PATTERN) -> list[Path]:
    found: dict[Path, Path] = {}
    for root in SEARCH_ROOTS:
        if root.exists():
            for path in root.glob(pattern):
                found[path.resolve()] = path.resolve()
    files = sorted(found.values(), key=lambda x: int(re.search(r"reg_(\d+)", x.name).group(1)))
    if not files:
        raise FileNotFoundError(
            f"No files matching {pattern!r} were found in {[str(p) for p in SEARCH_ROOTS]}"
        )
    return files


def reg_from_path(path: Path) -> int:
    match = re.search(r"reg_(\d+)", path.name)
    if not match:
        raise ValueError(f"Could not parse regularisation value from {path.name}")
    return int(match.group(1))


def style_figure(
    fig: go.Figure,
    *,
    height: int = 700,
    width: int | None = None,
    title: str | None = None,
    legend_y: float = 1.10,
    hovermode: str = "x unified",
) -> go.Figure:
    fig.update_layout(
        template="plotly_white",
        title={"text": title, "x": 0.5} if title else None,
        height=height,
        width=width,
        hovermode=hovermode,
        plot_bgcolor="white",
        paper_bgcolor="white",
        margin=dict(l=60, r=40, t=80, b=60),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=legend_y,
            xanchor="center",
            x=0.5,
            bgcolor="rgba(255,255,255,0.85)",
        ),
        font=dict(size=13),
    )
    fig.update_xaxes(showgrid=True, gridcolor="rgba(0,0,0,0.08)", zeroline=False)
    fig.update_yaxes(showgrid=True, gridcolor="rgba(0,0,0,0.08)", zeroline=False)
    return fig


def save_figure(fig: go.Figure, stem: str) -> None:
    html_path = OUTPUT_DIR / f"{stem}.html"
    fig.write_html(html_path, include_plotlyjs="cdn")
    try:
        fig.write_image(OUTPUT_DIR / f"{stem}.png", scale=2)
    except Exception as exc:
        print(f"[info] Static export skipped for {stem}: {exc}")


def equal_count_bucket_labels(n_buckets: int) -> list[str]:
    return [f"Q{i}" for i in range(1, n_buckets + 1)]


def bucketize_equal_count(series: pd.Series, n_buckets: int = 5) -> pd.Series:
    ranked = series.rank(method="first")
    return pd.qcut(ranked, q=n_buckets, labels=equal_count_bucket_labels(n_buckets))


In [3]:

files = find_calibration_files()
files


[PosixPath('/Users/nikitamahbub/Desktop/uni/thesis/SVCJ-Deribit-Inverse-Options-Pricer-main/excel files/calibration_results_reg_0.xlsx'),
 PosixPath('/Users/nikitamahbub/Desktop/uni/thesis/SVCJ-Deribit-Inverse-Options-Pricer-main/excel files/calibration_results_reg_1.xlsx'),
 PosixPath('/Users/nikitamahbub/Desktop/uni/thesis/SVCJ-Deribit-Inverse-Options-Pricer-main/excel files/calibration_results_reg_3.xlsx'),
 PosixPath('/Users/nikitamahbub/Desktop/uni/thesis/SVCJ-Deribit-Inverse-Options-Pricer-main/excel files/calibration_results_reg_5.xlsx'),
 PosixPath('/Users/nikitamahbub/Desktop/uni/thesis/SVCJ-Deribit-Inverse-Options-Pricer-main/excel files/calibration_results_reg_10.xlsx'),
 PosixPath('/Users/nikitamahbub/Desktop/uni/thesis/SVCJ-Deribit-Inverse-Options-Pricer-main/excel files/calibration_results_reg_25.xlsx'),
 PosixPath('/Users/nikitamahbub/Desktop/uni/thesis/SVCJ-Deribit-Inverse-Options-Pricer-main/excel files/calibration_results_reg_50.xlsx'),
 PosixPath('/Users/nikitamahbub

## 1. Load the regularisation sweep and summarize the SVCJ calibration diagnostics

In [4]:

def compute_regularisation_summary(files: list[Path]) -> pd.DataFrame:
    rows = []
    selected = ["kappa", "theta", "sigma_v", "rho", "v0", "lam", "ell_y", "sigma_y", "ell_v", "rho_j"]

    for path in files:
        reg = reg_from_path(path)
        svcj = pd.read_excel(path, sheet_name="svcj_params")
        svcj["timestamp"] = pd.to_datetime(svcj["timestamp"], utc=True)
        svcj["feller_gap"] = 2 * svcj["kappa"] * svcj["theta"] - svcj["sigma_v"] ** 2

        roughness_terms = []
        for _, group in svcj.sort_values("timestamp").groupby("currency"):
            for param in selected:
                series = group[param].astype(float)
                iqr = series.quantile(0.75) - series.quantile(0.25)
                scale = iqr if pd.notna(iqr) and iqr > 1e-12 else series.std()
                if pd.isna(scale) or scale <= 1e-12:
                    continue
                roughness_terms.append(float(series.diff().abs().median() / scale))

        rows.append(
            {
                "reg": reg,
                "rmse_test_mean": float(svcj["rmse_test"].mean()),
                "mae_test_mean": float(svcj["mae_test"].mean()),
                "nfev_mean": float(svcj["nfev"].mean()),
                "nfev_median": float(svcj["nfev"].median()),
                "roughness_mean": float(np.mean(roughness_terms)),
                "feller_violation_share": float((svcj["feller_gap"] < 0).mean()),
                "rho_boundary_share": float((svcj["rho"].abs() > 0.95).mean()),
                "rhoj_boundary_share": float((svcj["rho_j"].abs() > 0.95).mean()),
            }
        )

    out = pd.DataFrame(rows).sort_values("reg").reset_index(drop=True)
    out.to_csv(OUTPUT_DIR / "regularisation_summary.csv", index=False)
    return out


reg_summary = compute_regularisation_summary(files)
reg_summary


,reg,rmse_test_mean,mae_test_mean,nfev_mean,nfev_median,roughness_mean,feller_violation_share,rho_boundary_share,rhoj_boundary_share
0,0,0.002929,0.001293,45.125000,28.0,0.124487,0.439433,0.289948,0.181701
1,1,0.002890,0.001279,26.237113,17.5,0.112663,0.479381,0.118557,0.323454
2,3,0.002895,0.001280,24.143041,15.0,0.107055,0.465206,0.118557,0.268041
3,5,0.002898,0.001283,22.045103,15.0,0.096080,0.449742,0.117268,0.266753
4,10,0.002905,0.001292,23.023196,13.0,0.086405,0.427835,0.117268,0.266753
5,25,0.002930,0.001308,13.903351,10.0,0.053812,0.295103,0.288660,0.251289
6,50,0.002907,0.001297,13.686856,9.0,0.067559,0.350515,0.117268,0.266753
7,100,0.002911,0.001304,11.712629,8.0,0.061768,0.326031,0.117268,0.266753
8,250,0.002961,0.001343,9.471649,7.0,0.030549,0.243557,0.333763,0.251289
9,500,0.002973,0.001361,9.315722,6.0,0.025786,0.239691,0.333763,0.251289


In [5]:
fig = make_subplots(
    rows=1,
    cols=2,
    horizontal_spacing=0.20,
    specs=[[{"secondary_y": True}, {"secondary_y": False}]],
    # subplot_titles=(
    #     "Average out-of-sample fit and optimisation cost",
    #     "Aggregate parameter-path roughness",
    # ),
)

# Left panel: fit + optimisation cost
fig.add_trace(
    go.Scatter(
        x=reg_summary["reg"],
        y=reg_summary["rmse_test_mean"],
        mode="lines+markers+text",
        name="Mean test RMSE",
        text=reg_summary["reg"].astype(str),
        textposition="top center",
        textfont=dict(size=10),
        cliponaxis=False,
        line=dict(color="#111827", width=2),
        marker=dict(size=8),
        hovertemplate="λ_TS=%{x}<br>Mean test RMSE=%{y:.6f}<extra></extra>",
    ),
    row=1,
    col=1,
    secondary_y=False,
)

fig.add_trace(
    go.Scatter(
        x=reg_summary["reg"],
        y=reg_summary["nfev_mean"],
        mode="lines+markers",
        name="Mean function evaluations",
        line=dict(color="#2563eb", width=2, dash="dash"),
        marker=dict(size=8),
        hovertemplate="λ_TS=%{x}<br>Mean function evaluations=%{y:.2f}<extra></extra>",
    ),
    row=1,
    col=1,
    secondary_y=True,
)

# Right panel: roughness
fig.add_trace(
    go.Scatter(
        x=reg_summary["reg"],
        y=reg_summary["roughness_mean"],
        mode="lines+markers+text",
        name="Roughness index",
        text=reg_summary["reg"].astype(str),
        textposition="top center",
        textfont=dict(size=10),
        cliponaxis=False,
        line=dict(color="#dc2626", width=2),
        marker=dict(size=8),
        hovertemplate="λ_TS=%{x}<br>Roughness index=%{y:.4f}<extra></extra>",
    ),
    row=1,
    col=2,
)

# Final-choice marker in both panels
for col in [1, 2]:
    fig.add_vline(
        x=FINAL_REG,
        line_width=1.5,
        line_dash="dot",
        line_color="#dc2626",
        row=1,
        col=col,
    )

# X-axes
tickvals = reg_summary["reg"].tolist()
ticktext = [str(x) for x in tickvals]

fig.update_xaxes(
    type="log",
    tickvals=tickvals,
    ticktext=ticktext,
    title_text="λ_TS (log scale)",
    row=1,
    col=1,
    showgrid=True,
)
fig.update_xaxes(
    type="log",
    tickvals=tickvals,
    ticktext=ticktext,
    title_text="λ_TS (log scale)",
    row=1,
    col=2,
    showgrid=True,
)

# Y-axes
fig.update_yaxes(
    title_text="Mean test RMSE",
    row=1,
    col=1,
    secondary_y=False,
    tickformat=".5f",
    showgrid=True,
)
fig.update_yaxes(
    title_text="Mean function evaluations",
    row=1,
    col=1,
    secondary_y=True,
    showgrid=False,
)
fig.update_yaxes(
    title_text="Roughness index",
    row=1,
    col=2,
    showgrid=True,
)

style_figure(
    fig,
    height=520,
    title="Regularisation trade-off across the SVCJ sweep",
    legend_y=1.10,
)

fig.update_layout(
    margin=dict(t=95, b=70, l=70, r=70),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5,
    ),
    hovermode="x unified",
)

save_figure(fig, "regularisation_tradeoff")
fig.show()

/opt/miniconda3/lib/python3.13/site-packages/kaleido/_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.




[info] Static export skipped for regularisation_tradeoff: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



## 2. Load the baseline (`lambda_TS = 100`) and build snapshot-level pricing diagnostics

In [6]:

baseline_path = next(path for path in files if reg_from_path(path) == FINAL_REG)

black_params = pd.read_excel(baseline_path, sheet_name="black_params")
heston_params = pd.read_excel(baseline_path, sheet_name="heston_params")
svcj_params = pd.read_excel(baseline_path, sheet_name="svcj_params")

for frame in [black_params, heston_params, svcj_params]:
    frame["timestamp"] = pd.to_datetime(frame["timestamp"], utc=True)

test_cols = [
    "snapshot_ts",
    "currency",
    "time_to_maturity",
    "moneyness",
    "bid_price",
    "ask_price",
    "mid_price_clean",
    "price_black",
    "price_heston",
    "price_svcj",
]
test_data = pd.read_excel(baseline_path, sheet_name="test_data", usecols=test_cols)
test_data["snapshot_ts"] = pd.to_datetime(test_data["snapshot_ts"], utc=True)

for model in ["black", "heston", "svcj"]:
    error = test_data[f"price_{model}"] - test_data["mid_price_clean"]
    test_data[f"abs_err_{model}"] = error.abs()
    test_data[f"sq_err_{model}"] = error ** 2
    test_data[f"within_{model}"] = (
        (test_data[f"price_{model}"] >= test_data["bid_price"])
        & (test_data[f"price_{model}"] <= test_data["ask_price"])
    ).astype(int)

test_data.head()


,snapshot_ts,currency,time_to_maturity,bid_price,ask_price,mid_price_clean,price_black,price_heston,price_svcj,moneyness,abs_err_black,sq_err_black,within_black,abs_err_heston,sq_err_heston,within_heston,abs_err_svcj,sq_err_svcj,within_svcj
0,2025-12-30 17:31:15+00:00,BTC,0.00439,0.0005,0.0007,0.00060,0.000372,0.000231,0.000546,0.948604,0.000228,5.198118e-08,0,0.000369,1.365110e-07,0,0.000054,2.945227e-09,1
1,2025-12-30 17:31:15+00:00,BTC,0.00439,0.0009,0.0012,0.00105,0.001079,0.000736,0.000886,0.959897,0.000029,8.362172e-10,1,0.000314,9.880662e-08,0,0.000164,2.680168e-08,0
2,2025-12-30 17:31:15+00:00,BTC,0.00439,0.0290,0.0320,0.03050,0.031334,0.030700,0.030609,0.971190,0.000834,6.950971e-07,1,0.000200,3.998427e-08,1,0.000109,1.196730e-08,1
3,2025-12-30 17:31:15+00:00,BTC,0.00439,0.0065,0.0070,0.00675,0.008587,0.007324,0.006746,0.993776,0.001837,3.375879e-06,0,0.000574,3.289488e-07,0,0.000004,1.482094e-11,1
4,2025-12-30 17:31:15+00:00,BTC,0.00439,0.0008,0.0011,0.00095,0.001328,0.000767,0.000719,1.038948,0.000378,1.431710e-07,0,0.000183,3.349997e-08,0,0.000231,5.350183e-08,0


In [7]:

def compute_overall_baseline_metrics(test_data: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for currency, frame in [("BTC", test_data.query("currency == 'BTC'")), ("ETH", test_data.query("currency == 'ETH'")), ("Pooled", test_data)]:
        row = {"currency": currency, "n_test_options": int(len(frame))}
        for model in ["black", "heston", "svcj"]:
            row[f"rmse_{model}"] = float(np.sqrt(frame[f"sq_err_{model}"].mean()))
            row[f"mae_{model}"] = float(frame[f"abs_err_{model}"].mean())
            row[f"within_{model}"] = float(frame[f"within_{model}"].mean())
        rows.append(row)
    out = pd.DataFrame(rows)
    out.to_csv(OUTPUT_DIR / "baseline_overall_metrics.csv", index=False)
    return out


def compute_snapshot_metrics(test_data: pd.DataFrame) -> pd.DataFrame:
    rows = []
    grouped = test_data.groupby(["currency", "snapshot_ts"], as_index=False)
    for (currency, ts), frame in grouped:
        row = {
            "currency": currency,
            "snapshot_ts": ts,
            "n_options": int(len(frame)),
        }
        for model in ["black", "heston", "svcj"]:
            row[f"rmse_{model}"] = float(np.sqrt(frame[f"sq_err_{model}"].mean()))
            row[f"mae_{model}"] = float(frame[f"abs_err_{model}"].mean())
            row[f"within_{model}"] = float(frame[f"within_{model}"].mean())
        rows.append(row)
    out = pd.DataFrame(rows).sort_values(["currency", "snapshot_ts"]).reset_index(drop=True)
    out.to_csv(OUTPUT_DIR / "snapshot_time_series_metrics.csv", index=False)
    return out


baseline_overall = compute_overall_baseline_metrics(test_data)
snapshot_metrics = compute_snapshot_metrics(test_data)

baseline_overall


,currency,n_test_options,rmse_black,mae_black,within_black,rmse_heston,mae_heston,within_heston,rmse_svcj,mae_svcj,within_svcj
0,BTC,57494,0.005330,0.003565,0.256670,0.002892,0.001482,0.452030,0.002546,0.001004,0.618830
1,ETH,49532,0.006151,0.003929,0.321368,0.004485,0.002112,0.490915,0.004227,0.001626,0.631047
2,Pooled,107026,0.005724,0.003733,0.286613,0.003715,0.001774,0.470026,0.003429,0.001292,0.624484


In [8]:

fig = make_subplots(
    rows=2,
    cols=2,
    shared_xaxes=True,
    vertical_spacing=0.10,
    horizontal_spacing=0.08,
    subplot_titles=("BTC RMSE", "ETH RMSE", "BTC MAE", "ETH MAE"),
)

panel_map = {
    (1, 1): ("BTC", "rmse"),
    (1, 2): ("ETH", "rmse"),
    (2, 1): ("BTC", "mae"),
    (2, 2): ("ETH", "mae"),
}

for (row, col), (currency, metric) in panel_map.items():
    frame = snapshot_metrics.query("currency == @currency").sort_values("snapshot_ts")
    for model in ["black", "heston", "svcj"]:
        fig.add_trace(
            go.Scatter(
                x=frame["snapshot_ts"],
                y=frame[f"{metric}_{model}"],
                mode="lines",
                name=MODEL_LABELS[model],
                legendgroup=MODEL_LABELS[model],
                showlegend=(row, col) == (1, 1),
                line=dict(color=MODEL_COLORS[model], width=2),
            ),
            row=row,
            col=col,
        )

fig.update_yaxes(title_text="RMSE", row=1, col=1)
fig.update_yaxes(title_text="RMSE", row=1, col=2)
fig.update_yaxes(title_text="MAE", row=2, col=1)
fig.update_yaxes(title_text="MAE", row=2, col=2)

style_figure(fig, height=650)
save_figure(fig, "time_series_rmse_mae_panel")
fig.show()


[info] Static export skipped for time_series_rmse_mae_panel: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



In [20]:

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.10,
    subplot_titles=("BTC within-spread proportion", "ETH within-spread proportion"),
)

for idx, currency in enumerate(["BTC", "ETH"], start=1):
    frame = snapshot_metrics.query("currency == @currency").sort_values("snapshot_ts")
    for model in ["black", "heston", "svcj"]:
        fig.add_trace(
            go.Scatter(
                x=frame["snapshot_ts"],
                y=frame[f"within_{model}"],
                mode="lines",
                name=MODEL_LABELS[model],
                legendgroup=MODEL_LABELS[model],
                showlegend=(idx == 1),
                line=dict(color=MODEL_COLORS[model], width=2),
            ),
            row=idx,
            col=1,
        )
    fig.update_yaxes(title_text="Share inside spread", tickformat=".0%", row=idx, col=1)

style_figure(fig, height=600)
save_figure(fig, "time_series_within_spread_panel")
fig.show()


[info] Static export skipped for time_series_within_spread_panel: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



## 3. Build cross-sectional bucket diagnostics by equal-count maturity and moneyness bins

In [10]:

def compute_bucket_metrics(test_data: pd.DataFrame, n_buckets: int = 5) -> pd.DataFrame:
    rows = []

    for currency, frame in test_data.groupby("currency"):
        frame = frame.copy()

        for dimension in ["time_to_maturity", "moneyness"]:
            bucket_name = f"{dimension}_bucket"
            frame[bucket_name] = bucketize_equal_count(frame[dimension], n_buckets=n_buckets)

            bucket_ranges = (
                frame.groupby(bucket_name, observed=False)[dimension]
                .agg(["min", "max", "median", "count"])
                .reset_index()
            )

            for bucket, bucket_frame in frame.groupby(bucket_name, observed=False):
                bounds = bucket_ranges.loc[bucket_ranges[bucket_name] == bucket].iloc[0]
                row = {
                    "currency": currency,
                    "dimension": dimension,
                    "bucket": bucket,
                    "n_options": int(len(bucket_frame)),
                    "bucket_min": float(bounds["min"]),
                    "bucket_median": float(bounds["median"]),
                    "bucket_max": float(bounds["max"]),
                }
                for model in ["black", "heston", "svcj"]:
                    row[f"rmse_{model}"] = float(np.sqrt(bucket_frame[f"sq_err_{model}"].mean()))
                    row[f"mae_{model}"] = float(bucket_frame[f"abs_err_{model}"].mean())
                    row[f"within_{model}"] = float(bucket_frame[f"within_{model}"].mean())
                rows.append(row)

    out = pd.DataFrame(rows)
    out.to_csv(OUTPUT_DIR / "cross_sectional_bucket_metrics.csv", index=False)
    return out


bucket_metrics = compute_bucket_metrics(test_data, n_buckets=5)
bucket_metrics.head()


,currency,dimension,bucket,n_options,bucket_min,bucket_median,bucket_max,rmse_black,mae_black,within_black,rmse_heston,mae_heston,within_heston,rmse_svcj,mae_svcj,within_svcj
0,BTC,time_to_maturity,Q1,11499,0.003051,0.008059,0.023091,0.002692,0.001670,0.314114,0.001845,0.001018,0.420384,0.001702,0.000841,0.456040
1,BTC,time_to_maturity,Q2,11499,0.023091,0.042928,0.071028,0.003398,0.002268,0.301070,0.002257,0.001093,0.492043,0.002121,0.000833,0.614575
2,BTC,time_to_maturity,Q3,11498,0.071028,0.128523,0.186012,0.003871,0.002942,0.237085,0.002305,0.001233,0.456601,0.002040,0.000740,0.699774
3,BTC,time_to_maturity,Q4,11499,0.186012,0.272847,0.504131,0.005243,0.003930,0.224193,0.003205,0.001703,0.397774,0.002775,0.000957,0.669884
4,BTC,time_to_maturity,Q5,11499,0.504131,0.733595,0.998369,0.008986,0.007016,0.206888,0.004210,0.002366,0.493347,0.003624,0.001648,0.653883


In [25]:
def make_bucket_axis_labels(frame: pd.DataFrame, dimension: str) -> list[str]:
    if dimension == "time_to_maturity":
        # show bucket ranges in days for readability
        lows = (365.0 * frame["bucket_min"]).round().astype(int)
        highs = (365.0 * frame["bucket_max"]).round().astype(int)
        meds = (365.0 * frame["bucket_median"]).round().astype(int)

        labels = []
        for low, high, med in zip(lows, highs, meds):
            labels.append(f"{low}\u2013{high}d<br>(med {med}d)")
        return labels

    # fallback for moneyness buckets
    return [
        f"{low:.2f}\u2013{high:.2f}<br>(med {med:.2f})"
        for low, high, med in zip(frame["bucket_min"], frame["bucket_max"], frame["bucket_median"])
    ]


def make_bucket_hover_text(frame: pd.DataFrame, dimension: str) -> list[str]:
    if dimension == "time_to_maturity":
        return [
            (
                f"Maturity bucket<br>"
                f"median T = {med*365:.1f} days<br>"
                f"range = [{low*365:.1f}, {high*365:.1f}] days"
            )
            for med, low, high in zip(
                frame["bucket_median"], frame["bucket_min"], frame["bucket_max"]
            )
        ]

    return [
        f"Moneyness bucket<br>median K/F0 = {med:.3f}<br>range = [{low:.3f}, {high:.3f}]"
        for med, low, high in zip(
            frame["bucket_median"], frame["bucket_min"], frame["bucket_max"]
        )
    ]


def plot_bucket_panel(bucket_metrics: pd.DataFrame, dimension: str, stem: str, title: str) -> go.Figure:
    fig = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=False,
        vertical_spacing=0.12,
        subplot_titles=("BTC", "ETH"),
    )

    for row_idx, currency in enumerate(["BTC", "ETH"], start=1):
        frame = (
            bucket_metrics.query("currency == @currency and dimension == @dimension")
            .copy()
            .sort_values("bucket_median")
        )

        frame["bucket_label"] = make_bucket_axis_labels(frame, dimension)
        hover_text = make_bucket_hover_text(frame, dimension)

        for model in ["black", "heston", "svcj"]:
            fig.add_trace(
                go.Bar(
                    x=frame["bucket_label"],
                    y=frame[f"rmse_{model}"],
                    name=MODEL_LABELS[model],
                    legendgroup=MODEL_LABELS[model],
                    showlegend=(row_idx == 1),
                    marker_color=MODEL_COLORS[model],
                    hovertext=hover_text,
                    hovertemplate="%{hovertext}<br>RMSE = %{y:.6f}<extra></extra>",
                ),
                row=row_idx,
                col=1,
            )

    fig.update_layout(barmode="group")

    fig.update_yaxes(title_text="RMSE", row=1, col=1)
    fig.update_yaxes(title_text="RMSE", row=2, col=1)

    fig.update_xaxes(tickangle=0, row=1, col=1)
    fig.update_xaxes(tickangle=0, row=2, col=1)

    style_figure(fig, height=600, title=title)
    save_figure(fig, stem)
    return fig


fig = plot_bucket_panel(
    bucket_metrics,
    dimension="time_to_maturity",
    stem="bucket_diagnostics_maturity",
    title="",
)
fig.show()

[info] Static export skipped for bucket_diagnostics_maturity: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



In [26]:

fig = plot_bucket_panel(
    bucket_metrics,
    dimension="moneyness",
    stem="bucket_diagnostics_moneyness",
    title="",
)
fig.show()


[info] Static export skipped for bucket_diagnostics_moneyness: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



## 4. Inspect SVCJ parameter stability and identification issues at the selected regularisation level

In [13]:

def compute_parameter_summary(svcj_params: pd.DataFrame) -> pd.DataFrame:
    frame = svcj_params.copy()
    frame["feller_gap"] = 2 * frame["kappa"] * frame["theta"] - frame["sigma_v"] ** 2

    rows = []
    for currency, group in frame.groupby("currency"):
        row = {"currency": currency}
        ordered = group.sort_values("timestamp")

        for param in ["kappa", "theta", "sigma_v", "rho", "v0", "lam", "ell_y", "sigma_y", "ell_v", "rho_j"]:
            row[f"{param}_mean"] = float(group[param].mean())
            row[f"{param}_median"] = float(group[param].median())
            row[f"{param}_std"] = float(group[param].std())
            row[f"{param}_iqr"] = float(group[param].quantile(0.75) - group[param].quantile(0.25))
            row[f"{param}_mad_diff"] = float(ordered[param].diff().abs().median())

        row["rho_boundary_share"] = float((group["rho"].abs() > 0.95).mean())
        row["rhoj_boundary_share"] = float((group["rho_j"].abs() > 0.95).mean())
        row["feller_violation_share"] = float((group["feller_gap"] < 0).mean())
        row["corr_theta_v0"] = float(group[["theta", "v0"]].corr().iloc[0, 1])
        row["corr_kappa_theta"] = float(group[["kappa", "theta"]].corr().iloc[0, 1])
        row["corr_lam_sigmay"] = float(group[["lam", "sigma_y"]].corr().iloc[0, 1])
        row["corr_lam_elly"] = float(group[["lam", "ell_y"]].corr().iloc[0, 1])
        rows.append(row)

    out = pd.DataFrame(rows)
    out.to_csv(OUTPUT_DIR / "svcj_parameter_summary.csv", index=False)
    return out


parameter_summary = compute_parameter_summary(svcj_params)
parameter_summary


,currency,kappa_mean,kappa_median,kappa_std,kappa_iqr,kappa_mad_diff,theta_mean,theta_median,theta_std,theta_iqr,theta_mad_diff,sigma_v_mean,sigma_v_median,sigma_v_std,sigma_v_iqr,sigma_v_mad_diff,rho_mean,rho_median,rho_std,rho_iqr,rho_mad_diff,v0_mean,v0_median,v0_std,v0_iqr,v0_mad_diff,lam_mean,lam_median,lam_std,lam_iqr,lam_mad_diff,ell_y_mean,ell_y_median,ell_y_std,ell_y_iqr,ell_y_mad_diff,sigma_y_mean,sigma_y_median,sigma_y_std,sigma_y_iqr,sigma_y_mad_diff,ell_v_mean,ell_v_median,ell_v_std,ell_v_iqr,ell_v_mad_diff,rho_j_mean,rho_j_median,rho_j_std,rho_j_iqr,rho_j_mad_diff,rho_boundary_share,rhoj_boundary_share,feller_violation_share,corr_theta_v0,corr_kappa_theta,corr_lam_sigmay,corr_lam_elly
0,BTC,18.321268,13.376951,15.243121,23.473255,1.260094,0.105365,0.108179,0.044961,0.078675,0.004061,1.140263,0.940205,0.997941,1.339445,0.035805,-0.503349,-0.425279,0.297321,0.510608,0.027512,0.234256,0.224903,0.130760,0.171309,0.012783,1.483631,1.287897,0.793252,1.170157,0.06686,-0.155155,-0.067601,0.235725,0.253921,0.025211,0.167111,0.157563,0.094250,0.100285,7.262488e-03,1.159608,0.737491,1.062112,1.377052,0.050908,-0.026105,-0.030299,0.186512,0.188284,0.021043,0.118557,0.000000,0.090206,0.487601,0.445377,-0.138888,0.516223
1,ETH,21.141514,15.700853,15.665017,26.032807,1.261543,0.242251,0.248903,0.081989,0.096752,0.007597,2.544790,2.269343,1.446716,2.345601,0.113056,0.000635,0.142201,0.429771,0.307624,0.034981,0.386074,0.377010,0.188679,0.230505,0.019687,2.483239,2.271351,1.642230,1.354114,0.07546,-0.102021,-0.131923,0.175634,0.169802,0.024280,0.020327,0.000278,0.055977,0.000730,1.487254e-10,1.551592,0.334262,2.767811,0.896295,0.021720,-0.532854,-0.998999,0.501120,0.984304,0.000001,0.115979,0.533505,0.561856,-0.181787,0.187039,0.708814,0.282268


In [14]:

def plot_parameter_paths(
    svcj_params: pd.DataFrame,
    currency: str,
    stem: str,
    title: str,
    params: list[str] | None = None,
) -> go.Figure:
    params = params or ["v0", "theta", "sigma_v", "rho", "lam", "ell_y", "sigma_y", "rho_j"]
    labels = {
        "v0": "v0",
        "theta": "theta",
        "sigma_v": "sigma_v",
        "rho": "rho",
        "lam": "lambda",
        "ell_y": "ell_y",
        "sigma_y": "sigma_y",
        "rho_j": "rho_j",
    }

    frame = svcj_params.query("currency == @currency").sort_values("timestamp").copy()
    n_rows = math.ceil(len(params) / 2)
    fig = make_subplots(
        rows=n_rows,
        cols=2,
        shared_xaxes=True,
        vertical_spacing=0.08,
        horizontal_spacing=0.10,
        subplot_titles=[labels[p] for p in params],
    )

    color = "#b45309" if currency == "BTC" else "#0f766e"

    for idx, param in enumerate(params):
        row = idx // 2 + 1
        col = idx % 2 + 1
        fig.add_trace(
            go.Scatter(
                x=frame["timestamp"],
                y=frame[param],
                mode="lines",
                name=currency,
                showlegend=(idx == 0),
                line=dict(color=color, width=2),
            ),
            row=row,
            col=col,
        )
        if param in {"rho", "rho_j"}:
            fig.add_hline(y=0.95, line_dash="dot", line_color="#9ca3af", row=row, col=col)
            fig.add_hline(y=-0.95, line_dash="dot", line_color="#9ca3af", row=row, col=col)

    style_figure(fig, height=350 * n_rows, title=title, hovermode="x unified")
    save_figure(fig, stem)
    return fig


fig_btc = plot_parameter_paths(
    svcj_params,
    currency="BTC",
    stem="svcj_parameter_paths_btc",
    title="Selected SVCJ parameter paths for BTC at lambda_TS = 100",
)
fig_btc.show()


[info] Static export skipped for svcj_parameter_paths_btc: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



In [15]:

fig_eth = plot_parameter_paths(
    svcj_params,
    currency="ETH",
    stem="svcj_parameter_paths_eth",
    title="Selected SVCJ parameter paths for ETH at lambda_TS = 100",
)
fig_eth.show()


[info] Static export skipped for svcj_parameter_paths_eth: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido




## 5. Compact headline diagnostics for the text draft

These cells provide a few numbers that are convenient when writing the prose around the figures.


In [16]:

baseline_overall


,currency,n_test_options,rmse_black,mae_black,within_black,rmse_heston,mae_heston,within_heston,rmse_svcj,mae_svcj,within_svcj
0,BTC,57494,0.005330,0.003565,0.256670,0.002892,0.001482,0.452030,0.002546,0.001004,0.618830
1,ETH,49532,0.006151,0.003929,0.321368,0.004485,0.002112,0.490915,0.004227,0.001626,0.631047
2,Pooled,107026,0.005724,0.003733,0.286613,0.003715,0.001774,0.470026,0.003429,0.001292,0.624484


In [17]:

candidate_regs = reg_summary.query("reg in [1, 25, 50, 100, 250, 500, 1000]").copy()
candidate_regs


,reg,rmse_test_mean,mae_test_mean,nfev_mean,nfev_median,roughness_mean,feller_violation_share,rho_boundary_share,rhoj_boundary_share
1,1,0.002890,0.001279,26.237113,17.5,0.112663,0.479381,0.118557,0.323454
5,25,0.002930,0.001308,13.903351,10.0,0.053812,0.295103,0.288660,0.251289
6,50,0.002907,0.001297,13.686856,9.0,0.067559,0.350515,0.117268,0.266753
7,100,0.002911,0.001304,11.712629,8.0,0.061768,0.326031,0.117268,0.266753
8,250,0.002961,0.001343,9.471649,7.0,0.030549,0.243557,0.333763,0.251289
9,500,0.002973,0.001361,9.315722,6.0,0.025786,0.239691,0.333763,0.251289
10,1000,0.002995,0.001389,8.921392,6.0,0.021708,0.221649,0.333763,0.251289


In [18]:

for currency in ["BTC", "ETH"]:
    frame = snapshot_metrics.query("currency == @currency")
    print(currency)
    print("SVCJ beats Heston on RMSE in", (frame["rmse_svcj"] < frame["rmse_heston"]).mean())
    print("SVCJ beats Heston on MAE in", (frame["mae_svcj"] < frame["mae_heston"]).mean())
    print("SVCJ beats Heston on within-spread in", (frame["within_svcj"] > frame["within_heston"]).mean())
    print()


BTC
SVCJ beats Heston on RMSE in 0.961340206185567
SVCJ beats Heston on MAE in 0.979381443298969
SVCJ beats Heston on within-spread in 0.9742268041237113

ETH
SVCJ beats Heston on RMSE in 0.9252577319587629
SVCJ beats Heston on MAE in 0.9742268041237113
SVCJ beats Heston on within-spread in 0.9381443298969072

